# # PET REMOTE SYSTEM_(4) AUTOML2 + 누수 제거 + inference

In [1]:
#!/usr/bin/env python3
"""
Jupyter용 반려동물 테스트 데이터 생성 스크립트
사용법: 
- create_test_data()           # 기본 20개
- create_test_data(50)         # 50개 생성
- create_test_data(30, True)   # 30개 생성 후 바탕화면에 저장
"""

import pandas as pd
import numpy as np
import random
import os
from pathlib import Path

def get_desktop_path():
    """바탕화면 경로 찾기 (Windows/Mac/Linux 지원)"""
    desktop_paths = [
        os.path.join(os.path.expanduser("~"), "Desktop"),      # Windows/Mac/Linux
        os.path.join(os.path.expanduser("~"), "바탕 화면"),     # Windows 한글
        os.path.join(os.path.expanduser("~"), "바탕화면"),      # Windows 한글 (다른 버전)
        str(Path.home() / "Desktop"),                          # pathlib 방식
    ]
    
    for path in desktop_paths:
        if os.path.exists(path):
            return path
    
    return "."  # 바탕화면 못 찾으면 현재 폴더

def create_test_pet_data(n_samples=20, save_to_desktop=False):
    """
    Jupyter용 테스트 데이터 생성 함수
    
    Parameters:
    - n_samples: 생성할 샘플 수 (기본 20)
    - save_to_desktop: True시 바탕화면에 저장 (기본 False)
    
    Returns:
    - DataFrame: 생성된 테스트 데이터
    """
    
    print(f"🐕 ===== Jupyter용 테스트 데이터 생성 =====")
    print(f"📊 생성할 샘플 수: {n_samples}")
    print(f"💾 저장 위치: {'바탕화면' if save_to_desktop else '현재 폴더'}")
    
    # 품종 목록 (원본 데이터와 유사하게)
    breeds = [
        '푸들', '말티즈', '비숑', '치와와', '포메라니안', 
        '말티푸', '포메', '믹스', '비숑프리제', '시츄',
        '골든리트리버', '닥스훈트', '요크셔테리어', '허스키', '복서'
    ]
    
    # 나이 카테고리
    age_categories = ['유령견', '성견', '노령견']
    
    # 크기 카테고리  
    size_categories = ['소형견', '중형견', '대형견']
    
    data = []
    
    for i in range(n_samples):
        # 기본 정보
        pet_data = {
            'pet_id': f'TEST_PET_{i+1:04d}',
            'dog_type': random.choice(breeds),
            'age': random.choice(age_categories),
            'size': random.choice(size_categories),
            'timestamp': 1725523870 + i * 3600,  # 1시간씩 증가
            'age_numeric': random.randint(1, 3),
            'size_numeric': random.randint(1, 3),
            'timestamp_normalized': random.random()
        }
        
        # 포즈 특성 생성 (P0~P11, P12는 제외)
        for joint in range(12):  # P0 ~ P11
            # 가끔 결측값 생성 (현실적으로)
            if random.random() > 0.85:  # 15% 확률로 결측
                pet_data[f'P{joint}_x'] = np.nan
                pet_data[f'P{joint}_y'] = np.nan
                pet_data[f'P{joint}_conf'] = np.nan
                pet_data[f'P{joint}_label'] = np.nan
            else:
                # 정상적인 포즈 값들
                pet_data[f'P{joint}_x'] = random.uniform(0.1, 0.9)
                pet_data[f'P{joint}_y'] = random.uniform(0.1, 0.9)
                pet_data[f'P{joint}_conf'] = random.uniform(0.7, 1.0)
                pet_data[f'P{joint}_label'] = random.randint(0, 12)
        
        # 센서 특성 생성
        # 건강한 개체와 아픈 개체의 차이를 시뮬레이션
        is_healthy = random.random() > 0.3  # 70% 정상, 30% 질병
        
        if is_healthy:
            # 정상적인 센서 값들
            base_variance = random.uniform(0.5, 2.0)
            noise_level = random.uniform(0.1, 0.5)
        else:
            # 비정상적인 센서 값들 (더 높은 변동성)
            base_variance = random.uniform(2.0, 5.0)
            noise_level = random.uniform(0.5, 1.5)
        
        # 센서 통계값들
        for sensor_type in ['sensor', 'sensor_0', 'sensor_1', 'sensor_2']:
            pet_data[f'{sensor_type}_mean'] = random.uniform(5, 15) + (0 if is_healthy else random.uniform(2, 8))
            pet_data[f'{sensor_type}_std'] = base_variance + random.uniform(-0.5, 0.5)
            pet_data[f'{sensor_type}_min'] = pet_data[f'{sensor_type}_mean'] - random.uniform(2, 5)
            pet_data[f'{sensor_type}_max'] = pet_data[f'{sensor_type}_mean'] + random.uniform(2, 5)
            pet_data[f'{sensor_type}_median'] = pet_data[f'{sensor_type}_mean'] + random.uniform(-1, 1)
        
        # 추가 센서 특성들
        pet_data['sensor_range'] = pet_data['sensor_max'] - pet_data['sensor_min']
        pet_data['sensor_var'] = pet_data['sensor_std'] ** 2
        
        # 엔지니어링된 특성들
        pet_data['pose_confidence_mean'] = random.uniform(0.7, 0.95)
        pet_data['pose_confidence_std'] = random.uniform(0.05, 0.2)
        pet_data['sensor_instability'] = pet_data['sensor_std'] + random.uniform(0, 1)
        pet_data['sensor_stability'] = 1 / (pet_data['sensor_instability'] + 0.01)
        pet_data['pose_detection_rate'] = random.uniform(0.8, 1.0)
        pet_data['pose_x_range'] = random.uniform(0.3, 0.8)
        pet_data['pose_y_range'] = random.uniform(0.3, 0.8)
        pet_data['pose_area'] = pet_data['pose_x_range'] * pet_data['pose_y_range']
        
        # 품종별 정규화 (가상의 값)
        pet_data['age_numeric_breed_norm'] = random.uniform(-1, 1)
        pet_data['size_numeric_breed_norm'] = random.uniform(-1, 1)
        
        data.append(pet_data)
    
    # DataFrame 생성
    df = pd.DataFrame(data)
    
    # 일부 컬럼 정수형으로 변환
    int_cols = ['age_numeric', 'size_numeric', 'timestamp']
    for col in int_cols:
        if col in df.columns:
            df[col] = df[col].astype(int)
    
    print(f"✅ 데이터 생성 완료: {df.shape}")
    print(f"📊 포함된 특성:")
    print(f"   - 기본 정보: pet_id, dog_type, age, size")
    print(f"   - 포즈 특성: P0~P11 (x, y, conf, label)")
    print(f"   - 센서 특성: sensor_* (mean, std, min, max 등)")
    print(f"   - 엔지니어링 특성: pose_confidence_*, sensor_stability 등")
    
    # 파일 저장
    output_file = f"test_pet_data_{n_samples}.csv"
    
    # 저장 경로 결정
    if save_to_desktop:
        desktop_path = get_desktop_path()
        output_path = os.path.join(desktop_path, output_file)
    else:
        output_path = output_file
    
    try:
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\n💾 테스트 데이터 저장: {output_path}")
        
        # 미리보기
        print(f"\n📋 데이터 미리보기:")
        display_cols = ['pet_id', 'dog_type', 'age', 'size']
        print(df[display_cols].head())
        
        # 건강 상태 분포 (시뮬레이션)
        healthy_count = sum(1 for i in range(n_samples) if random.random() > 0.3)
        disease_count = n_samples - healthy_count
        print(f"\n📊 시뮬레이션된 건강 상태:")
        print(f"   🟢 정상 예상: ~{healthy_count}마리 ({healthy_count/n_samples*100:.1f}%)")
        print(f"   🔴 질병 예상: ~{disease_count}마리 ({disease_count/n_samples*100:.1f}%)")
        
        print(f"\n🔮 추론 실행 방법:")
        print(f"   results = run_quick_inference('{output_file}')")
        print(f"   # 또는")
        print(f"   inference = PetHealthInference('saved_model_fixed_pet_data_1050')")
        print(f"   results = inference.run_inference('{output_file}')")
        
        print(f"\n✅ 테스트 데이터 생성 완료!")
        
        return df
        
    except Exception as e:
        print(f"❌ 파일 저장 실패: {e}")
        print(f"💡 데이터는 생성되었지만 파일 저장에 문제가 있습니다.")
        return df

def create_test_data_quick(n=20):
    """빠른 테스트 데이터 생성 (짧은 함수명)"""
    return create_test_pet_data(n, save_to_desktop=False)

def create_test_data_desktop(n=20):
    """바탕화면에 테스트 데이터 생성"""
    return create_test_pet_data(n, save_to_desktop=True)

def show_usage():
    """사용법 안내"""
    print("🐕 Jupyter용 테스트 데이터 생성기 사용법")
    print("=" * 45)
    print()
    print("📋 기본 사용법:")
    print("   df = create_test_pet_data()           # 20개 생성")
    print("   df = create_test_pet_data(50)         # 50개 생성")
    print("   df = create_test_pet_data(30, True)   # 30개 생성 후 바탕화면 저장")
    print()
    print("⚡ 빠른 사용법:")
    print("   df = create_test_data_quick(100)      # 100개 현재폴더 저장")
    print("   df = create_test_data_desktop(50)     # 50개 바탕화면 저장")
    print()
    print("🔮 추론 실행:")
    print("   results = run_quick_inference()       # 자동으로 테스트 파일 찾아서 실행")
    print()
    print("💡 권장 워크플로우:")
    print("   1. df = create_test_data_quick(50)    # 테스트 데이터 생성")
    print("   2. results = run_quick_inference()    # 추론 실행")
    print("   3. # 결과 분석...")

# Jupyter에서 바로 실행할 수 있는 기본 함수들
def create_test_data(n_samples=20, save_to_desktop=False):
    """메인 함수 (기본 실행용)"""
    return create_test_pet_data(n_samples, save_to_desktop)

# 즉시 실행 가능한 예시들
def demo_small():
    """소규모 데모 (10마리)"""
    print("🎬 소규모 데모 실행...")
    return create_test_pet_data(10, False)

def demo_medium():
    """중간 규모 데모 (50마리)"""
    print("🎬 중간 규모 데모 실행...")
    return create_test_pet_data(50, False)

def demo_large():
    """대규모 데모 (100마리)"""
    print("🎬 대규모 데모 실행...")
    return create_test_pet_data(100, True)  # 바탕화면에 저장

# 자동 실행 (Jupyter에서 import시)
if __name__ == "__main__":
    show_usage()
else:
    # Jupyter에서 import될 때 사용법 간단히 표시
    print("🐕 테스트 데이터 생성기 로드됨!")
    print("💡 사용법: create_test_data(개수) 또는 show_usage() 참고")

🐕 Jupyter용 테스트 데이터 생성기 사용법

📋 기본 사용법:
   df = create_test_pet_data()           # 20개 생성
   df = create_test_pet_data(50)         # 50개 생성
   df = create_test_pet_data(30, True)   # 30개 생성 후 바탕화면 저장

⚡ 빠른 사용법:
   df = create_test_data_quick(100)      # 100개 현재폴더 저장
   df = create_test_data_desktop(50)     # 50개 바탕화면 저장

🔮 추론 실행:
   results = run_quick_inference()       # 자동으로 테스트 파일 찾아서 실행

💡 권장 워크플로우:
   1. df = create_test_data_quick(50)    # 테스트 데이터 생성
   2. results = run_quick_inference()    # 추론 실행
   3. # 결과 분석...


In [2]:
# 바탕화면 경로 자동 탐지 기능 추가
def find_desktop_test_files():
    """바탕화면에서 테스트 데이터 파일 찾기"""
    import os
    from pathlib import Path
    
    # 바탕화면 경로 찾기 (Windows/Mac/Linux 지원)
    desktop_paths = [
        os.path.join(os.path.expanduser("~"), "Desktop"),      # Windows/Mac/Linux
        os.path.join(os.path.expanduser("~"), "바탕 화면"),     # Windows 한글
        os.path.join(os.path.expanduser("~"), "바탕화면"),      # Windows 한글 (다른 버전)
        str(Path.home() / "Desktop"),                          # pathlib 방식
    ]
    
    # 바탕화면에서 테스트 파일 찾기
    for desktop_path in desktop_paths:
        if os.path.exists(desktop_path):
            test_files = [f for f in os.listdir(desktop_path) 
                         if f.startswith('test_pet_data') and f.endswith('.json')]
            if test_files:
                full_path = os.path.join(desktop_path, test_files[0])
                print(f"🖥️ 바탕화면에서 발견: {test_files[0]}")
                return full_path
    
    return None

# run_quick_inference 함수 수정 (data_file 찾기 부분)
def run_quick_inference(data_file=None, model_dir=None):
    """Jupyter에서 빠른 추론 실행 (바탕화면 지원)"""
    
    # 기본값 설정
    if data_file is None:
        # 1. 현재 폴더에서 찾기
        test_files = [f for f in os.listdir('.') if f.startswith('test_pet_data') and f.endswith('.csv')]
        if test_files:
            data_file = test_files[0]
        else:
            # 2. 바탕화면에서 찾기
            data_file = find_desktop_test_files()

In [5]:
# 바탕화면 경로 자동 탐지 기능 수정
def find_desktop_test_files():
    """바탕화면에서 테스트 데이터 파일 찾기"""
    import os
    from pathlib import Path
    
    # 바탕화면 경로 찾기 (Windows/Mac/Linux 지원)
    desktop_paths = [
        os.path.join(os.path.expanduser("~"), "Desktop"),      # Windows/Mac/Linux
        os.path.join(os.path.expanduser("~"), "바탕 화면"),     # Windows 한글
        os.path.join(os.path.expanduser("~"), "바탕화면"),      # Windows 한글 (다른 버전)
        str(Path.home() / "Desktop"),                          # pathlib 방식
    ]
    
    # 바탕화면에서 테스트 파일 찾기
    for desktop_path in desktop_paths:
        if os.path.exists(desktop_path):
            # 🐛 수정: .json -> .csv로 변경
            test_files = [f for f in os.listdir(desktop_path) 
                         if f.startswith('test_pet_data') and f.endswith('.csv')]
            if test_files:
                full_path = os.path.join(desktop_path, test_files[0])
                print(f"🖥️ 바탕화면에서 발견: {test_files[0]}")
                return full_path
    
    return None

# run_quick_inference 함수 개선
def run_quick_inference_fixed(data_file=None, model_dir=None):
    """Jupyter에서 빠른 추론 실행 (바탕화면 지원, 개선버전)"""
    
    # 기본값 설정
    if data_file is None:
        # 1. 현재 폴더에서 찾기
        test_files = [f for f in os.listdir('.') if f.startswith('test_pet_data') and f.endswith('.csv')]
        if test_files:
            data_file = test_files[0]
            print(f"📁 현재 폴더에서 발견: {data_file}")
        else:
            # 2. 바탕화면에서 찾기
            data_file = find_desktop_test_files()
        
        # 3. 여전히 없으면 에러
        if data_file is None:
            print("❌ 테스트 데이터 파일이 없습니다!")
            print("💡 먼저 테스트 데이터를 생성하세요:")
            print("   df = create_test_data_quick(50)")
            return None
        
    if model_dir is None:
        model_dirs = [d for d in os.listdir('.') if d.startswith('saved_model') and os.path.isdir(d)]
        model_dir = model_dirs[0] if model_dirs else None
    
    if model_dir is None:
        print("❌ 모델 폴더가 없습니다!")
        print("💡 먼저 모델을 훈련하세요:")
        print("exec(open('pet_automl_realistic.py').read())")
        return None
    
    print(f"🔮 빠른 추론 시작...")
    print(f"📄 데이터: {data_file}")
    print(f"📁 모델: {model_dir}")
    
    # 추론 실행
    try:
        inference = PetHealthInference(model_dir)
        results = inference.run_inference(data_file)
        return results
    except Exception as e:
        print(f"❌ 추론 실행 중 오류: {e}")
        return None

# 완전 자동화된 테스트 실행 함수
def auto_test_pipeline(n_samples=50):
    """테스트 데이터 생성부터 추론까지 한번에 실행"""
    print("🚀 자동 테스트 파이프라인 시작!")
    
    # 1. 테스트 데이터 생성
    print("\n1️⃣ 테스트 데이터 생성...")
    df = create_test_data_quick(n_samples)
    if df is None:
        print("❌ 테스트 데이터 생성 실패")
        return None
    
    # 2. 추론 실행
    print("\n2️⃣ 추론 실행...")
    results = run_quick_inference_fixed()
    
    if results:
        print("\n🎉 전체 파이프라인 완료!")
        return results
    else:
        print("❌ 파이프라인 실패")
        return None